In [0]:
%run ./00_config

### Build a historical taxi demand baseline by pickup zone, day of week, hour, and month

In [0]:
from pyspark.sql import functions as F

# Gold table: zone_hour_demand_baseline
# Source: nyc_taxi_company.silver.trips_clean
# Target: nyc_taxi_company.gold.zone_hour_demand_baseline
# Physical Delta path: abfss://taxicompany@azinfrasummit.dfs.core.windows.net/curated/gold/zone_hour_demand_baseline

silver_trips_table = f"{SILVER_SCHEMA}.trips_clean"

gold_baseline_table = f"{GOLD_SCHEMA}.zone_hour_demand_baseline"
gold_baseline_path = f"{CURATED_PATH}/gold/zone_hour_demand_baseline"

print(f"Reading Silver trips table: {silver_trips_table}")
print(f"Writing Gold baseline table: {gold_baseline_table}")
print(f"Gold baseline path: {gold_baseline_path}")

In [0]:
#read silver trip

trips_df = spark.table(silver_trips_table)

display(trips_df.limit(10))
trips_df.printSchema()

In [0]:
#create hourly zone demand

zone_hour_daily_df = (
    trips_df
    .groupBy(
        "pickup_date",
        "pickup_month",
        "pickup_day_of_week",
        "pickup_hour",
        "pickup_location_id",
        "pickup_borough",
        "pickup_zone",
        "pickup_service_zone",
        "pickup_is_airport",
        "pickup_is_manhattan"
    )
    .agg(
        F.count("*").alias("trip_count"),
        F.countDistinct("trip_id").alias("distinct_trip_count"),
        F.avg("fare_amount").alias("avg_fare_amount"),
        F.avg("total_amount").alias("avg_total_amount"),
        F.avg("trip_distance_miles").alias("avg_trip_distance_miles"),
        F.avg("trip_duration_minutes").alias("avg_trip_duration_minutes"),
        F.avg("avg_speed_mph").alias("avg_speed_mph"),
        F.sum("total_amount").alias("total_revenue_amount")
    )
    .withColumn("_gold_updated_ts", F.current_timestamp())
)

display(zone_hour_daily_df.limit(20))

In [0]:
#create historical baseline

# COMMAND ----------

zone_hour_baseline_df = (
    zone_hour_daily_df
    .groupBy(
        "pickup_month",
        "pickup_day_of_week",
        "pickup_hour",
        "pickup_location_id",
        "pickup_borough",
        "pickup_zone",
        "pickup_service_zone",
        "pickup_is_airport",
        "pickup_is_manhattan"
    )
    .agg(
        F.countDistinct("pickup_date").alias("observed_days"),
        F.avg("trip_count").alias("avg_trip_count"),
        F.expr("percentile_approx(trip_count, 0.5)").alias("median_trip_count"),
        F.expr("percentile_approx(trip_count, 0.9)").alias("p90_trip_count"),
        F.min("trip_count").alias("min_trip_count"),
        F.max("trip_count").alias("max_trip_count"),

        F.avg("avg_fare_amount").alias("avg_fare_amount"),
        F.avg("avg_total_amount").alias("avg_total_amount"),
        F.avg("avg_trip_distance_miles").alias("avg_trip_distance_miles"),
        F.avg("avg_trip_duration_minutes").alias("avg_trip_duration_minutes"),
        F.avg("avg_speed_mph").alias("avg_speed_mph"),

        F.avg("total_revenue_amount").alias("avg_total_revenue_amount")
    )
    .withColumn(
        "day_type",
        F.when(F.col("pickup_day_of_week").isin(1, 7), F.lit("Weekend"))
         .otherwise(F.lit("Weekday"))
    )
    .withColumn(
        "time_of_day",
        F.when(F.col("pickup_hour").between(5, 10), F.lit("Morning"))
         .when(F.col("pickup_hour").between(11, 15), F.lit("Midday"))
         .when(F.col("pickup_hour").between(16, 20), F.lit("Evening"))
         .otherwise(F.lit("Late Night"))
    )
    .withColumn("_gold_updated_ts", F.current_timestamp())
)

display(zone_hour_baseline_df.limit(20))

In [0]:
#write gold delta table to ADLS and register the external location on UC

(
    zone_hour_baseline_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_baseline_path)
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_baseline_table}
USING DELTA
LOCATION '{gold_baseline_path}'
""")

spark.sql(f"""
COMMENT ON TABLE {gold_baseline_table} IS
'Gold table containing historical taxi demand baselines by pickup zone, month, day of week, and hour. Used for BI, Genie, and live anomaly detection.'
""")

print(f"Registered or confirmed table: {gold_baseline_table}")

In [0]:
#validation checks

display(spark.sql(f"""
SELECT COUNT(*) AS row_count
FROM {gold_baseline_table}
"""))

display(spark.sql(f"""
SELECT
  pickup_borough,
  COUNT(*) AS baseline_rows,
  ROUND(AVG(avg_trip_count), 2) AS avg_hourly_trip_count,
  ROUND(MAX(p90_trip_count), 2) AS max_p90_trip_count
FROM {gold_baseline_table}
GROUP BY pickup_borough
ORDER BY avg_hourly_trip_count DESC
"""))

display(spark.sql(f"""
SELECT
  pickup_borough,
  pickup_zone,
  pickup_day_of_week,
  pickup_hour,
  observed_days,
  ROUND(avg_trip_count, 2) AS avg_trip_count,
  median_trip_count,
  p90_trip_count,
  ROUND(avg_total_amount, 2) AS avg_total_amount
FROM {gold_baseline_table}
WHERE pickup_borough = 'Manhattan'
ORDER BY avg_trip_count DESC
LIMIT 20
"""))

display(spark.sql(f"""
SELECT
  time_of_day,
  day_type,
  ROUND(AVG(avg_trip_count), 2) AS avg_trip_count
FROM {gold_baseline_table}
GROUP BY time_of_day, day_type
ORDER BY time_of_day, day_type
"""))

display(spark.sql(f"""
DESCRIBE DETAIL {gold_baseline_table}
"""))

### Join actual daily weather with taxi demand and compare observed demand to historical baseline

In [0]:
# paths for ingestion
# Sources:
#   nyc_taxi_company.silver.trips_clean
#   nyc_taxi_company.silver.weather_daily
#   nyc_taxi_company.gold.zone_hour_demand_baseline
# Target:
#   nyc_taxi_company.gold.weather_demand_impact

silver_trips_table = f"{SILVER_SCHEMA}.trips_clean"
silver_weather_table = f"{SILVER_SCHEMA}.weather_daily"
gold_baseline_table = f"{GOLD_SCHEMA}.zone_hour_demand_baseline"

gold_weather_impact_table = f"{GOLD_SCHEMA}.weather_demand_impact"
gold_weather_impact_path = f"{CURATED_PATH}/gold/weather_demand_impact"

print(f"Reading trips: {silver_trips_table}")
print(f"Reading weather: {silver_weather_table}")
print(f"Reading baseline: {gold_baseline_table}")
print(f"Writing Gold weather impact table: {gold_weather_impact_table}")

In [0]:
#load tables for aggregation

trips_df = spark.table(silver_trips_table)
weather_df = spark.table(silver_weather_table)
baseline_df = spark.table(gold_baseline_table)

In [0]:
# aggregation and computation of gold metrics

daily_zone_hour_demand_df = (
    trips_df
    .groupBy(
        "pickup_date",
        "pickup_month",
        "pickup_day_of_week",
        "pickup_hour",
        "pickup_location_id",
        "pickup_borough",
        "pickup_zone"
    )
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("total_amount").alias("avg_total_amount"),
        F.avg("trip_duration_minutes").alias("avg_trip_duration_minutes"),
        F.avg("avg_speed_mph").alias("avg_speed_mph")
    )
)

baseline_select_df = (
    baseline_df
    .select(
        "pickup_month",
        "pickup_day_of_week",
        "pickup_hour",
        "pickup_location_id",
        F.col("avg_trip_count").alias("baseline_avg_trip_count"),
        F.col("p90_trip_count").alias("baseline_p90_trip_count")
    )
)

weather_demand_impact_df = (
    daily_zone_hour_demand_df
    .join(
        weather_df,
        daily_zone_hour_demand_df.pickup_date == weather_df.weather_date,
        how="left"
    )
    .join(
        baseline_select_df,
        on=[
            "pickup_month",
            "pickup_day_of_week",
            "pickup_hour",
            "pickup_location_id"
        ],
        how="left"
    )
    .withColumn(
        "demand_lift_pct",
        F.when(
            F.col("baseline_avg_trip_count") > 0,
            ((F.col("trip_count") - F.col("baseline_avg_trip_count")) / F.col("baseline_avg_trip_count")) * F.lit(100.0)
        )
    )
    .withColumn(
        "demand_lift_band",
        F.when(F.col("demand_lift_pct") >= 50, F.lit("High Increase"))
         .when(F.col("demand_lift_pct") >= 15, F.lit("Moderate Increase"))
         .when(F.col("demand_lift_pct") <= -30, F.lit("Large Decrease"))
         .otherwise(F.lit("Normal"))
    )
    .select(
        "pickup_date",
        "pickup_month",
        "pickup_day_of_week",
        "pickup_hour",
        "pickup_location_id",
        "pickup_borough",
        "pickup_zone",

        "weather_condition",
        "precipitation_inches",
        "snowfall_inches",
        "temp_avg_f",

        "trip_count",
        "baseline_avg_trip_count",
        "baseline_p90_trip_count",
        "demand_lift_pct",
        "demand_lift_band",

        "avg_total_amount",
        "avg_trip_duration_minutes",
        "avg_speed_mph"
    )
    .withColumn("_gold_updated_ts", F.current_timestamp())
)

display(weather_demand_impact_df.limit(20))

In [0]:
#write df to ADLS as a delta table and register external location on UC

(
    weather_demand_impact_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_weather_impact_path)
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_weather_impact_table}
USING DELTA
LOCATION '{gold_weather_impact_path}'
""")

spark.sql(f"""
COMMENT ON TABLE {gold_weather_impact_table} IS
'Gold table comparing taxi demand by zone and hour against historical baseline with daily NYC Central Park weather context.'
""")

print(f"Registered or confirmed table: {gold_weather_impact_table}")

In [0]:
#validations

display(spark.sql(f"""
SELECT
  weather_condition,
  COUNT(*) AS zone_hour_rows,
  ROUND(AVG(trip_count), 2) AS avg_observed_trip_count,
  ROUND(AVG(baseline_avg_trip_count), 2) AS avg_baseline_trip_count,
  ROUND(AVG(demand_lift_pct), 2) AS avg_demand_lift_pct,
  ROUND(AVG(avg_trip_duration_minutes), 2) AS avg_trip_duration_minutes,
  ROUND(AVG(avg_speed_mph), 2) AS avg_speed_mph
FROM {gold_weather_impact_table}
GROUP BY weather_condition
ORDER BY zone_hour_rows DESC
"""))

### Analyze continuous taxi pickup demand trend around sports events

In [0]:
# Grain:
#   event_id + venue catchment zone + relative hour bucket
# Window:
#   event_start - 4 hours through estimated_event_end + 4 hours

silver_trips_table = f"{SILVER_SCHEMA}.trips_clean"
silver_sports_events_table = f"{SILVER_SCHEMA}.sports_events"
venue_catchment_table = f"{SILVER_SCHEMA}.venue_zone_catchment"
gold_baseline_table = f"{GOLD_SCHEMA}.zone_hour_demand_baseline"

gold_event_trend_table = f"{GOLD_SCHEMA}.sports_event_demand_trend"
gold_event_trend_path = f"{CURATED_PATH}/gold/sports_event_demand_trend"

print(f"Reading trips: {silver_trips_table}")
print(f"Reading sports events: {silver_sports_events_table}")
print(f"Reading venue catchment: {venue_catchment_table}")
print(f"Reading baseline: {gold_baseline_table}")
print(f"Writing Gold event trend table: {gold_event_trend_table}")

In [0]:
#load the dataframes

trips_df = spark.table(silver_trips_table)
events_df = spark.table(silver_sports_events_table)
catchment_df = spark.table(venue_catchment_table)
baseline_df = spark.table(gold_baseline_table)

# OPTIMIZATION: Pre-aggregate trips to zone-hour-day level.
# Instead of joining event buckets against raw trips (expensive range join on 175M+ rows),
# we pre-compute hourly zone-level trip counts and join against that (~5M rows, equi-join).

daily_zone_hour_demand_df = (
    trips_df
    .groupBy(
        F.col("pickup_date").alias("demand_date"),
        F.col("pickup_hour").alias("demand_hour"),
        F.col("pickup_location_id").alias("demand_location_id")
    )
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("total_amount").alias("avg_total_amount"),
        F.avg("trip_duration_minutes").alias("avg_trip_duration_minutes"),
        F.avg("trip_distance_miles").alias("avg_trip_distance_miles")
    )
)

print(f"Pre-aggregated daily zone-hour demand prepared (equi-join optimization)")

In [0]:
# Create hourly buckets around each event.
# relative_hour_from_start:
#   -4, -3, -2, -1 = pre-event
#    0, 1, 2... = during/after depending on event duration and estimated end

events_with_buckets_df = (
    events_df
    .select(
        "event_id",
        "league",
        "team_name",
        "opponent_name",
        "event_name",
        "venue_name",
        "venue_borough",
        "event_start_ts_local",
        "estimated_event_end_ts_local",
        "event_duration_hours",
        "analysis_window_start_ts",
        "analysis_window_end_ts"
    )
    .withColumn(
        "bucket_start_ts",
        F.explode(
            F.expr("sequence(analysis_window_start_ts, analysis_window_end_ts - INTERVAL 1 HOURS, INTERVAL 1 HOURS)")
        )
    )
    .withColumn("bucket_end_ts", F.expr("bucket_start_ts + INTERVAL 1 HOURS"))
    .withColumn(
        "relative_hour_from_start",
        F.floor((F.unix_timestamp("bucket_start_ts") - F.unix_timestamp("event_start_ts_local")) / F.lit(3600))
    )
    .withColumn(
        "event_phase",
        F.when(F.col("bucket_end_ts") <= F.col("event_start_ts_local"), F.lit("Pre-event"))
         .when(F.col("bucket_start_ts") >= F.col("estimated_event_end_ts_local"), F.lit("Post-event"))
         .otherwise(F.lit("During-event"))
    )
    .withColumn("bucket_date", F.to_date("bucket_start_ts"))
    .withColumn("bucket_month", F.month("bucket_start_ts"))
    .withColumn("bucket_day_of_week", F.dayofweek("bucket_start_ts"))
    .withColumn("bucket_hour", F.hour("bucket_start_ts"))
)

display(events_with_buckets_df.orderBy("event_start_ts_local", "bucket_start_ts").limit(50))

In [0]:
# Join event-hour buckets with catchment zones, then with pre-aggregated demand.
# OPTIMIZATION: equi-join on (location_id, date, hour) replaces the expensive
# range join that previously scanned raw trips for each event-hour-zone bucket.

event_zone_buckets_df = (
    events_with_buckets_df
    .join(
        catchment_df.select(
            "venue_name",
            F.col("location_id").alias("venue_catchment_location_id"),
            F.col("borough").alias("venue_catchment_borough"),
            F.col("zone_name").alias("venue_catchment_zone"),
            "catchment_type"
        ),
        on="venue_name",
        how="inner"
    )
)

# Fast equi-join on (location_id, date, hour) instead of range join on raw trips
event_demand_matched_df = (
    event_zone_buckets_df
    .join(
        daily_zone_hour_demand_df,
        (F.col("venue_catchment_location_id") == F.col("demand_location_id")) &
        (F.col("bucket_date") == F.col("demand_date")) &
        (F.col("bucket_hour") == F.col("demand_hour")),
        how="left"
    )
    .withColumn("pickup_trip_count", F.coalesce(F.col("trip_count"), F.lit(0)))
)

display(event_demand_matched_df.limit(20))

In [0]:
# Compare observed demand against historical baseline and compute demand lift

baseline_select_df = (
    baseline_df
    .select(
        F.col("pickup_location_id").alias("baseline_location_id"),
        F.col("pickup_month").alias("baseline_month"),
        F.col("pickup_day_of_week").alias("baseline_day_of_week"),
        F.col("pickup_hour").alias("baseline_hour"),
        F.col("avg_trip_count").alias("baseline_avg_trip_count"),
        F.col("p90_trip_count").alias("baseline_p90_trip_count")
    )
)

event_trend_with_baseline_df = (
    event_demand_matched_df.alias("e")
    .join(
        baseline_select_df.alias("b"),
        (F.col("e.venue_catchment_location_id") == F.col("b.baseline_location_id")) &
        (F.col("e.bucket_month") == F.col("b.baseline_month")) &
        (F.col("e.bucket_day_of_week") == F.col("b.baseline_day_of_week")) &
        (F.col("e.bucket_hour") == F.col("b.baseline_hour")),
        how="left"
    )
    .withColumn(
        "demand_lift_pct",
        F.when(
            F.col("baseline_avg_trip_count") > 0,
            ((F.col("pickup_trip_count") - F.col("baseline_avg_trip_count")) / F.col("baseline_avg_trip_count")) * F.lit(100.0)
        )
    )
    .withColumn(
        "demand_lift_band",
        F.when(F.col("demand_lift_pct") >= 100, F.lit("Very High Increase"))
         .when(F.col("demand_lift_pct") >= 50, F.lit("High Increase"))
         .when(F.col("demand_lift_pct") >= 20, F.lit("Moderate Increase"))
         .when(F.col("demand_lift_pct") <= -30, F.lit("Decrease"))
         .otherwise(F.lit("Normal"))
    )
    .withColumn("_gold_updated_ts", F.current_timestamp())
)

display(event_trend_with_baseline_df.orderBy("event_start_ts_local", "bucket_start_ts").limit(50))

In [0]:
# Compute event-day averages for rest-of-day comparison.
# For each event + zone, compute the average and max hourly trip count
# across all hour buckets in the analysis window.

event_day_rest_avg_df = (
    event_demand_matched_df
    .groupBy("event_id", "venue_name", "venue_catchment_location_id")
    .agg(
        F.avg("pickup_trip_count").alias("event_day_avg_hourly_trip_count"),
        F.max("pickup_trip_count").alias("event_day_max_hourly_trip_count")
    )
)

# Final trend table with baseline and event-day comparisons
sports_event_demand_trend_df = (
    event_trend_with_baseline_df.alias("e")
    .join(
        event_day_rest_avg_df.alias("r"),
        (F.col("e.event_id") == F.col("r.event_id")) &
        (F.col("e.venue_name") == F.col("r.venue_name")) &
        (F.col("e.venue_catchment_location_id") == F.col("r.venue_catchment_location_id")),
        how="left"
    )
    .select(
        F.col("e.event_id"),
        F.col("e.league"),
        F.col("e.team_name"),
        F.col("e.opponent_name"),
        F.col("e.event_name"),
        F.col("e.venue_name"),
        F.col("e.venue_borough"),
        F.col("e.event_start_ts_local"),
        F.col("e.estimated_event_end_ts_local"),

        F.col("e.bucket_start_ts"),
        F.col("e.bucket_end_ts"),
        F.col("e.relative_hour_from_start"),
        F.col("e.event_phase"),
        F.col("e.bucket_month"),
        F.col("e.bucket_day_of_week"),
        F.col("e.bucket_hour"),

        F.col("e.venue_catchment_location_id"),
        F.col("e.venue_catchment_borough"),
        F.col("e.venue_catchment_zone"),
        F.col("e.catchment_type"),

        F.col("e.pickup_trip_count"),
        F.col("e.avg_total_amount"),
        F.col("e.avg_trip_duration_minutes"),
        F.col("e.avg_trip_distance_miles"),

        F.col("e.baseline_avg_trip_count"),
        F.col("e.baseline_p90_trip_count"),
        F.col("e.demand_lift_pct"),
        F.col("e.demand_lift_band"),

        F.col("r.event_day_avg_hourly_trip_count"),
        F.col("r.event_day_max_hourly_trip_count"),

        F.col("e._gold_updated_ts")
    )
    .withColumn(
        "event_day_lift_vs_avg_hour_pct",
        F.when(
            F.col("event_day_avg_hourly_trip_count") > 0,
            ((F.col("pickup_trip_count") - F.col("event_day_avg_hourly_trip_count")) / F.col("event_day_avg_hourly_trip_count")) * F.lit(100.0)
        )
    )
)

display(
    sports_event_demand_trend_df
    .orderBy("event_start_ts_local", "bucket_start_ts", "venue_catchment_zone")
    .limit(50)
)

In [0]:
#write trend table to ADLS and register external location on UC

# COMMAND ----------

(
    sports_event_demand_trend_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_event_trend_path)
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_event_trend_table}
USING DELTA
LOCATION '{gold_event_trend_path}'
""")

spark.sql(f"""
COMMENT ON TABLE {gold_event_trend_table} IS
'Gold table showing continuous hourly taxi pickup demand trends around curated sports events, compared to historical zone-hour baseline and event-day average.'
""")

print(f"Registered or confirmed table: {gold_event_trend_table}")

In [0]:
#validations

display(spark.sql(f"""
SELECT COUNT(*) AS row_count
FROM {gold_event_trend_table}
"""))

display(spark.sql(f"""
SELECT
  venue_name,
  event_phase,
  COUNT(*) AS rows,
  ROUND(AVG(pickup_trip_count), 2) AS avg_pickup_trips,
  ROUND(AVG(baseline_avg_trip_count), 2) AS avg_baseline_trips,
  ROUND(AVG(demand_lift_pct), 2) AS avg_demand_lift_pct,
  ROUND(AVG(event_day_lift_vs_avg_hour_pct), 2) AS avg_event_day_lift_pct
FROM {gold_event_trend_table}
GROUP BY venue_name, event_phase
ORDER BY venue_name, event_phase
"""))

display(spark.sql(f"""
SELECT
  league,
  team_name,
  venue_name,
  relative_hour_from_start,
  event_phase,
  ROUND(AVG(pickup_trip_count), 2) AS avg_pickup_trips,
  ROUND(AVG(demand_lift_pct), 2) AS avg_demand_lift_pct
FROM {gold_event_trend_table}
GROUP BY league, team_name, venue_name, relative_hour_from_start, event_phase
ORDER BY venue_name, relative_hour_from_start
"""))

### Analyze origin/destination flows around sports events

In [0]:
# Gold table: sports_event_od_flows
# Purpose:
# Analyze origin/destination flows around sports events:
#   - Pre-event inbound: trips ending near venue before event start
#   - Post-event outbound: trips starting near venue after estimated event end
# Grain:
#   event_id + flow_window + origin zone + destination zone

gold_event_od_table = f"{GOLD_SCHEMA}.sports_event_od_flows"
gold_event_od_path = f"{CURATED_PATH}/gold/sports_event_od_flows"

print(f"Writing Gold event OD flow table: {gold_event_od_table}")

In [0]:
#prepare trip od df

events_for_od_df = (
    events_df
    .select(
        "event_id",
        "league",
        "team_name",
        "opponent_name",
        "event_name",
        "venue_name",
        "venue_borough",
        "event_start_ts_local",
        "estimated_event_end_ts_local"
    )
)

event_catchment_df = (
    events_for_od_df
    .join(
        catchment_df.select(
            "venue_name",
            F.col("location_id").alias("venue_catchment_location_id"),
            F.col("borough").alias("venue_catchment_borough"),
            F.col("zone_name").alias("venue_catchment_zone"),
            "catchment_type"
        ),
        on="venue_name",
        how="inner"
    )
)

trips_od_df = (
    trips_df
    .select(
        "trip_id",
        "pickup_ts",
        "pickup_location_id",
        "pickup_borough",
        "pickup_zone",
        "dropoff_location_id",
        "dropoff_borough",
        "dropoff_zone",
        "total_amount",
        "trip_duration_minutes",
        "trip_distance_miles"
    )
)

In [0]:
#pre-event inbound flows

pre_event_inbound_df = (
    event_catchment_df.alias("e")
    .join(
        trips_od_df.alias("t"),
        (F.col("t.dropoff_location_id") == F.col("e.venue_catchment_location_id")) &
        (F.col("t.pickup_ts") >= F.expr("e.event_start_ts_local - INTERVAL 4 HOURS")) &
        (F.col("t.pickup_ts") < F.col("e.event_start_ts_local")),
        how="inner"
    )
    .select(
        F.col("e.event_id"),
        F.col("e.league"),
        F.col("e.team_name"),
        F.col("e.opponent_name"),
        F.col("e.event_name"),
        F.col("e.venue_name"),
        F.col("e.venue_borough"),
        F.col("e.event_start_ts_local"),
        F.col("e.estimated_event_end_ts_local"),
        F.lit("Pre-event inbound").alias("flow_window"),
        F.col("e.venue_catchment_location_id"),
        F.col("e.venue_catchment_zone"),
        F.col("e.catchment_type"),

        F.col("t.pickup_location_id").alias("origin_location_id"),
        F.col("t.pickup_borough").alias("origin_borough"),
        F.col("t.pickup_zone").alias("origin_zone"),

        F.col("t.dropoff_location_id").alias("destination_location_id"),
        F.col("t.dropoff_borough").alias("destination_borough"),
        F.col("t.dropoff_zone").alias("destination_zone"),

        F.col("t.trip_id"),
        F.col("t.total_amount"),
        F.col("t.trip_duration_minutes"),
        F.col("t.trip_distance_miles")
    )
)

In [0]:
# post-event outbound flows

post_event_outbound_df = (
    event_catchment_df.alias("e")
    .join(
        trips_od_df.alias("t"),
        (F.col("t.pickup_location_id") == F.col("e.venue_catchment_location_id")) &
        (F.col("t.pickup_ts") >= F.col("e.estimated_event_end_ts_local")) &
        (F.col("t.pickup_ts") < F.expr("e.estimated_event_end_ts_local + INTERVAL 4 HOURS")),
        how="inner"
    )
    .select(
        F.col("e.event_id"),
        F.col("e.league"),
        F.col("e.team_name"),
        F.col("e.opponent_name"),
        F.col("e.event_name"),
        F.col("e.venue_name"),
        F.col("e.venue_borough"),
        F.col("e.event_start_ts_local"),
        F.col("e.estimated_event_end_ts_local"),
        F.lit("Post-event outbound").alias("flow_window"),
        F.col("e.venue_catchment_location_id"),
        F.col("e.venue_catchment_zone"),
        F.col("e.catchment_type"),

        F.col("t.pickup_location_id").alias("origin_location_id"),
        F.col("t.pickup_borough").alias("origin_borough"),
        F.col("t.pickup_zone").alias("origin_zone"),

        F.col("t.dropoff_location_id").alias("destination_location_id"),
        F.col("t.dropoff_borough").alias("destination_borough"),
        F.col("t.dropoff_zone").alias("destination_zone"),

        F.col("t.trip_id"),
        F.col("t.total_amount"),
        F.col("t.trip_duration_minutes"),
        F.col("t.trip_distance_miles")
    )
)

In [0]:
#aggregate od flows

# COMMAND ----------

event_od_detail_df = pre_event_inbound_df.unionByName(post_event_outbound_df)

sports_event_od_flows_df = (
    event_od_detail_df
    .groupBy(
        "event_id",
        "league",
        "team_name",
        "opponent_name",
        "event_name",
        "venue_name",
        "venue_borough",
        "event_start_ts_local",
        "estimated_event_end_ts_local",
        "flow_window",
        "venue_catchment_location_id",
        "venue_catchment_zone",
        "catchment_type",
        "origin_location_id",
        "origin_borough",
        "origin_zone",
        "destination_location_id",
        "destination_borough",
        "destination_zone"
    )
    .agg(
        F.countDistinct("trip_id").alias("trip_count"),
        F.avg("total_amount").alias("avg_total_amount"),
        F.avg("trip_duration_minutes").alias("avg_trip_duration_minutes"),
        F.avg("trip_distance_miles").alias("avg_trip_distance_miles")
    )
    .withColumn(
        "flow_direction",
        F.when(F.col("flow_window") == "Pre-event inbound", F.lit("Inbound to venue"))
         .when(F.col("flow_window") == "Post-event outbound", F.lit("Outbound from venue"))
         .otherwise(F.lit("Other"))
    )
    .withColumn("_gold_updated_ts", F.current_timestamp())
)

display(sports_event_od_flows_df.orderBy(F.desc("trip_count")).limit(50))

In [0]:
# write od flows table to ADLS and register external location on UC

(
    sports_event_od_flows_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_event_od_path)
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_event_od_table}
USING DELTA
LOCATION '{gold_event_od_path}'
""")

spark.sql(f"""
COMMENT ON TABLE {gold_event_od_table} IS
'Gold table showing pre-event inbound and post-event outbound taxi origin-destination flows around curated sports events.'
""")

print(f"Registered or confirmed table: {gold_event_od_table}")

In [0]:
# validate od table

display(spark.sql(f"""
SELECT COUNT(*) AS row_count
FROM {gold_event_od_table}
"""))

display(spark.sql(f"""
SELECT
  venue_name,
  flow_window,
  SUM(trip_count) AS total_trips,
  COUNT(*) AS od_pairs
FROM {gold_event_od_table}
GROUP BY venue_name, flow_window
ORDER BY venue_name, flow_window
"""))

display(spark.sql(f"""
SELECT
  venue_name,
  flow_window,
  origin_borough,
  destination_borough,
  SUM(trip_count) AS total_trips
FROM {gold_event_od_table}
GROUP BY venue_name, flow_window, origin_borough, destination_borough
ORDER BY total_trips DESC
LIMIT 50
"""))

display(spark.sql(f"""
SELECT
  event_name,
  venue_name,
  flow_window,
  origin_zone,
  destination_zone,
  trip_count,
  ROUND(avg_total_amount, 2) AS avg_total_amount,
  ROUND(avg_trip_duration_minutes, 2) AS avg_trip_duration_minutes
FROM {gold_event_od_table}
ORDER BY trip_count DESC
LIMIT 50
"""))